### Imports

In [1]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [ ]:
# %pip install pyspark graphframes-py==0.10.0 graphframes-py==0.9.0 

In [ ]:
# %pip install --upgrade grpcio-status>=1.48.1 grpcio google protobuf pyarrow

In [2]:
!pip list | grep pyspark
!pip list | grep graphframes

pyspark                  4.0.1
graphframes-py           0.10.0


In [ ]:
# !pip list | findstr pyspark
# !pip list | findstr graphframes

In [3]:
!pyspark --version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/01 11:29:46 WARN Utils: Your hostname, Kenuey, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/01 11:29:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.1
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 17.0.18
Branch HEAD
Compiled by user runner on 2025-09-02T03:10:51Z
Revision 29434ea766b0fc3c3bf6eaadb43a8f931133649e
Url https://github.com/apache/spark
Type --help for more information.


In [4]:
# !pip install findspark networkx matplotlib pyvis

In [4]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop\hadoop-3.3.6"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\hadoop-3.3.6\bin"

# print(os.environ.get("HADOOP_HOME"))
# print(os.environ.get("PATH"))

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql.types import *

In [ ]:
# spark = SparkSession.builder \
#     .appName("GraphProject") \
#     .config("spark.jars.packages", "io.graphframes:graphframes-spark3_2.13:0.9.0-spark3.5") \
#     .getOrCreate()

26/06/01 11:22:45 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [7]:
spark = (
    SparkSession.builder
    .appName("AmazonGraph")
    .config("spark.jars.packages", "io.graphframes:graphframes-spark3_2.12:0.10.0")
    .getOrCreate()
)

print("Active Spark sessions:", spark.sparkContext.uiWebUrl)

Active Spark sessions: http://10.255.255.254:4040


In [17]:
AMAZONMETA_TXT = 'data/amazon-meta.txt'
RECOMMS_CSV = 'data/recomms.csv'
MEASURES_JSON = 'data/measures.json'

In [29]:
# !head -45 "data/amazon-meta.txt"

In [9]:
raw = spark.read.text(AMAZONMETA_TXT)

raw.show(20)

+--------------------+
|               value|
+--------------------+
|# Full informatio...|
| Total items: 548552|
|                    |
|             Id:   0|
|    ASIN: 0771044445|
|  discontinued pr...|
|                    |
|             Id:   1|
|    ASIN: 0827229534|
|  title: Patterns...|
|         group: Book|
|   salesrank: 396585|
|  similar: 5  080...|
|       categories: 2|
|   |Books[283155]...|
|   |Books[283155]...|
|  reviews: total:...|
|    2000-7-28  cu...|
|    2003-12-14  c...|
|                    |
+--------------------+
only showing top 20 rows


In [10]:
rdd = spark.sparkContext.textFile(AMAZONMETA_TXT)

In [28]:
rdd.take(50)

['# Full information about Amazon Share the Love products',
 'Total items: 548552',
 '',
 'Id:   0',
 'ASIN: 0771044445',
 '  discontinued product',
 '',
 'Id:   1',
 'ASIN: 0827229534',
 '  title: Patterns of Preaching: A Sermon Sampler',
 '  group: Book',
 '  salesrank: 396585',
 '  similar: 5  0804215715  156101074X  0687023955  0687074231  082721619X',
 '  categories: 2',
 '   |Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Preaching[12368]',
 '   |Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Sermons[12370]',
 '  reviews: total: 2  downloaded: 2  avg rating: 5',
 '    2000-7-28  cutomer: A2JW67OY8U6HHK  rating: 5  votes:  10  helpful:   9',
 '    2003-12-14  cutomer: A2VE83MZF98ITY  rating: 5  votes:   6  helpful:   5',
 '',
 'Id:   2',
 'ASIN: 0738700797',
 '  title: Candlemas: Feast of Flames',
 '  group: Book',
 '  salesrank: 168596',
 '  similar: 5  0738700827  1567184960  1567182836  0738

In [30]:
rdd.count()

15010574

In [31]:
rdd.getNumPartitions()

30

In [33]:
rdd.glom().map(len).collect()

[516610,
 514252,
 516218,
 514593,
 516118,
 515430,
 512380,
 517043,
 517669,
 515940,
 520971,
 517222,
 515816,
 515597,
 513376,
 518164,
 530414,
 533057,
 519084,
 520509,
 520588,
 519433,
 520296,
 525464,
 522565,
 523157,
 507377,
 491980,
 460324,
 58927]

### Data Preprocessing

In [34]:
from pyspark.sql.types import *
from pyspark.sql import Row

# 1. Разбиваем на блоки
def split_blocks(iter):
    block = []
    for line in iter:
        line = line.strip()
        if line.startswith("Id:") and block:
            yield block
            block = []
        block.append(line)
    if block:
        yield block

In [22]:
# 2. Парсим блок продукта
def parse_block(block):
    product = {
        "id": None,
        "asin": None,
        "title": "",
        "group": None,
        "salesrank": None,
        "similar_count": 0,
        "similar_asins": []
    }

    for line in block:
        if line.startswith("Id:"):
            product["id"] = line.split()[1]
        elif line.startswith("ASIN:"):
            product["asin"] = line.split()[1]
        elif line.startswith("title:"):
            product["title"] = line.replace("title:", "").strip()
        elif line.startswith("group:"):
            product["group"] = line.split(":")[1].strip()
        elif line.startswith("salesrank:"):
            try:
                product["salesrank"] = int(line.split(":")[1].strip())
            except:
                product["salesrank"] = None
        elif line.startswith("similar:"):
            parts = line.split()
            product["similar_count"] = int(parts[1])
            product["similar_asins"] = parts[2:]

    return [product]

In [23]:
# 3. rdd → blocks → parsed dicts
blocks_rdd = rdd.mapPartitions(split_blocks)
parsed_rdd = blocks_rdd.flatMap(parse_block)

In [24]:
# 4. схема
schema = StructType([
    StructField("id", StringType(), True),
    StructField("asin", StringType(), True),
    StructField("title", StringType(), True),
    StructField("group", StringType(), True),
    StructField("salesrank", IntegerType(), True),
    StructField("similar_count", IntegerType(), True),
    StructField("similar_asins", ArrayType(StringType()), True)
])

In [25]:
products_df = spark.createDataFrame(parsed_rdd, schema)
products_df.show(10, truncate=False)

+----+----------+-----------------------------------------------------------------+-----+---------+-------------+------------------------------------------------------------+
|id  |asin      |title                                                            |group|salesrank|similar_count|similar_asins                                               |
+----+----------+-----------------------------------------------------------------+-----+---------+-------------+------------------------------------------------------------+
|NULL|NULL      |                                                                 |NULL |NULL     |0            |[]                                                          |
|0   |0771044445|                                                                 |NULL |NULL     |0            |[]                                                          |
|1   |0827229534|Patterns of Preaching: A Sermon Sampler                          |Book |396585   |5            |[0804215715,

In [64]:
vertices = products_df.select(
    products_df.asin.alias("id"),
    "title",
    "group",
    "salesrank"
).distinct()


In [65]:
from pyspark.sql.functions import explode, col

edges = products_df \
    .withColumn("similar", explode("similar_asins")) \
    .select(
        col("asin").alias("src"),
        col("similar").alias("dst")
    )


In [66]:
from graphframes import GraphFrame

g = GraphFrame(vertices, edges)


In [67]:
# Vertices
vertices = products_df.selectExpr("asin as id", "title")

# Edges
from pyspark.sql.functions import explode, col

edges = products_df \
    .withColumn("dst", explode(col("similar_asins"))) \
    .select(col("asin").alias("src"), col("dst"))

edges.show(5)


+----------+----------+
|       src|       dst|
+----------+----------+
|0827229534|0804215715|
|0827229534|156101074X|
|0827229534|0687023955|
|0827229534|0687074231|
|0827229534|082721619X|
+----------+----------+
only showing top 5 rows


In [ ]:
vertices = products_df.selectExpr("asin as id", "title", "group", "salesrank")


edges = products_df.rdd.flatMap(lambda row: [
    Row(src=row.asin, dst=asin) for asin in row.similar_asins
]).toDF()


In [68]:
g = GraphFrame(vertices, edges)

# Пример: показать вершины и ребра
print("Vertices:")
g.vertices.show(5, truncate=False)

print("Edges:")
g.edges.show(5, truncate=False)


Vertices:
+----------+------------------------------------------------+
|id        |title                                           |
+----------+------------------------------------------------+
|NULL      |                                                |
|0771044445|                                                |
|0827229534|Patterns of Preaching: A Sermon Sampler         |
|0738700797|Candlemas: Feast of Flames                      |
|0486287785|World War II Allied Fighter Planes Trading Cards|
+----------+------------------------------------------------+
only showing top 5 rows
Edges:
+----------+----------+
|src       |dst       |
+----------+----------+
|0827229534|0804215715|
|0827229534|156101074X|
|0827229534|0687023955|
|0827229534|0687074231|
|0827229534|082721619X|
+----------+----------+
only showing top 5 rows


In [ ]:
# Пример: количество соседей для каждого товара
g.degrees.show(5)

# PageRank (важность товара в графе)
results = g.pageRank(resetProbability=0.15, maxIter=5)
results.vertices.select("id", "pagerank").show(5)


### Descriptive Analysis

### Bundles and Collections

### Graph Visualization

### New Recommender System